# GwenLand glcuda - T4 Ceiling Wave 3 (Direct Model Fetch)

Decision-grade Kaggle A/B: Wave 2 `r256` versus Wave 3 `grid.y` token parallelism on the same candidate binary. The notebook fetches the pinned Qwen2.5-0.5B GGUF directly, verifies its SHA-256, assembles PTX, runs hardware correctness, measures two paired production sessions, and packages one evidence ZIP.


## 1 · Configuration, T4 gate, and isolated source trees

The first code cell is intentionally strict. It rejects a non-T4 GPU, verifies compute capability 7.5, checks out the exact baseline commit twice, and applies the embedded candidate patch only to the candidate tree.


In [ ]:
import base64
import datetime as dt
import gzip
import hashlib
import json
import math
import os
from pathlib import Path
import re
import shutil
import statistics
import subprocess
import sys
import time

REPO_URL = "https://github.com/gwenland-org/gwenland-ai.git"
BASE_REV = "3bce8dd7b8aaa2765855ab927c611b54981f9241"
PATCH_SHA256 = "5f09f6147636c36db4e23b9d5f16a3384ca4c0b69679d5443d7c9a396387a508"
PATCH_GZIP_B64 = """H4sIAAAAAAAEAO1de3PbSHL/X59iVil7SZOEiAefWl1WtrV7Lj/WJynxXSkKBZIgiRUeNABKYvZ8lQ+RT5hPkn4MXgSoBeNLlRmdalekgZnGTE8/ft0zQk/t2Uy0WnM7EubR3JmspuZRGEyObq3As5xQXhqFbq+jLKMHMa7Q6MD2ptaDaI9NfaKaitKfjqeD6VSo7XbXMA5arValZx00Go1qz/vxR9HSus2eaNDvH388EEdHwrqzgrW4N4Ml/ApFYLUCy5za3lxEC0ucfbh8c34mzElk35mR7XsidMyxmAW+K95pAv5tR6Hw7z3RImozP6B+czOyxM9n7983haZ3+0Q/FA9C0wbi7UtxIv7WMdri/Uvhz5BOFJizmT0RSysQjrnyJouEmhn3cc0osB8Uceo4IiYIvU0xdvzJrQgXZmDRs0PThS/+reWFTRH6eO2gBdSAeOu2xa1zE7Inlqh1DRH49zhGXRM4Qg0eWhemN6Upwu2Zrh00dqBj9ImOXkYHZ5ftOzEdKxQ4A8+/FxeXpz+fvRZvPoiLP56ew7f3Z+9/Of8LsBuegDySD/emRGjie+HKtaZivM4s51CcJnydBj5wq//QFD/99IHWJRRzX7w+P33fGvsrb6ogIZIQnSREz0gI/vypP2qLC/9UfA5FGIGEuOLNBXDWDmG91v4qErVPOO+Wa/4Ki3ZyIl7++yUMzOELdUVSei1q/Yc+LIuu1UX8cwLryesFXIOltez5IiIucrePwPBogWIJgmfOXcsDRtbmgb9avnkNvR3Ts46MpojsufzXM6M+pDUXwmzDtdC13JF5VYvaoiFkx/oLWOiGWJjO7IXahW/Q/YUhFKVhXAtR8z1LrGCUtOSPkoF1rkCG5z9GMp+uvAyB66vb8aNDEUUy03ZTTFUg9ZdWZDsW8krAqBrxmJrI+lBoL4BQkz8aKpCJ7n1ifbLaHbWpqqKBH1q83o41N526Ii4ic452ANZ4vjKDKQhY5MOjQFz6NQ8WrE5rRDrm+SAL0BYXKRRja+1L2USFXJpTMiggpyCQk4iFDZfn3Pq8sgOLVnQoUIyegQCA9MD8bA/+AXzhf+HzxA8nqD81pBmsPA8UYbJYebch0TIj8fH87Kc3796NXp5evvqjwMb1Y/Ew+hwePYykipFuuqswgkGK03fvfnl1egkKtlrC3Gipqw7pWFjmZCHWrVeXp2LimC4oGCi45cyQS2ZExFwfHiQtQlPcL3CtdhgPMjDH75r1ABzkTmgskN3AG1R6cR/YUWR5oGrIjXdkRYcgZPZUTCzbqeFUjkD9xAtgpj09AjGoo0XudAUrV8g9L9CQTodCa5MBK1i4BnV5WTRfCk059+Da1ic3eVC0rEe4UIWxILGzlMXgY8Degxp0jRYZC/JECj1HWZ+oKKYoGH5gg9SajvQjTCeek97uaTvOCdeg9ff7ORDKnR3aYxAEBUQMbPXcGc0t1x25rjn63K8dkMIrSzMAG6usQHiWo/vPYZPUtaejce4ZTdUgbaWmgTUXyhjE8lnwg9H/w3H28gwvz37QtfzlMZB9Fkx/MAy43qLr5D+nQjEde+4JsELKuA8GD8wdCsL1MTY6wpViU51xlm/B70RWeNCQTT6Zd5bQh8Iz0SfB8+wwsoJQTEwPzISAB5i8VN7KHVv40AV6R9MBp0a8V5hUdmKj+XoUgQbyN28cfwNFPc615onhrQe/GX8N069r/zgZ6IezP1/Cosc+5UiqZIxhQn8WgSO1Wkt7aTm2BwO9bTm+vyyOb4w02l5TflO93KBoEe7DCTWAz43bS2TBs6UHuh1f374YKMFbF4MUFq0t9F3aEaKoEnJGTO0hvAKxZ2qSHA61oAQHguk4UxZLkkrgp9oUVySc18cknf1OE5zXQG9q3UQ60z60jhp1sb3r49LbOt1Go4ANChLl+ncM8RZgV78HJGIFNmg6WAR0gIRMcX3AToMJZsOggEEfmyG4n4QaEkArI0Fii3A4csyHfiioKI1k0dFpkYVyzVsQDMJWCRnbAyC0mhCfxpaDAwCS1gOwz1nnTRE+3rFaNMDUCSbjaaPtMsELOJG9BMMAqBbdHA4DyAQWCKQl+vUYzMIzQKPI22Z8Q0IuXb7WpuMFZXRNGLcIVwgLbRB6ibFBaOfo303bEWBt5eCA33JhEv2bQBNgqxTUcOGUKih+6x4nAE/O8kTI3uAIuoZ8xMpR7u2pJZ+T0dyUonYsGR71Qchxqf3ZLLQipgBzVEIWSL2Z/mZCyTiDzETYfIAodjJDjMdJIs8AOySsTU/cPthwY7D8gCx/UpMU5k2Sdpx5LmoeP/uRCRrN9DdTOd4+svXGyNTyQa2zxrFkUOC9l6vosVF1munvnI0NV2MlTNgOwtdkHY9HFQ/efCg2S74mrWxPttJz97tGfhUJM91TUMohEUUpINQHDbY5qVQjIwGQKA/SGqViQpMxSiSE6GN4NbKn3AnUVOpAV3bS1UIvhNUQl5CZVI0BenG115Mx1gY7NVqtab/J/5Bjm9xFaHfjBdZUkuHe8bbu8UdmLDCIueOP0SABtAdcHgVgZaZ3JoSVIcR82tHtuM5wIFVsjcahdZAXqaw5fmzQk9tGPzuW7M1+bqgp+7VBk5zatn4D7p1jJmooezOeA3SCScxsMOB15m632xwAdwcGfOS4y7JDfNM1+q1uIf0QTvKEpQOUsSglS2TT87PT1wIcjAXBQgwk0IBTROa+6GcCRWWTtToJsrrFEMXx7QuBeYcC49PORoFDaVejX2AA9ZO9u5s9ZejJI93sxAui6wWuQVgsTikYkWxzMUkA5ghhdT1jL76OmN436rGiJjw0YjZoBTHie3KtjdIH4UqXPgpjZJYnwDOAajRQOAlrCmLcRiMlyn7gMbdjUXsrEzaENOoZYPMxsF1LvPqX8/OzD9ug6O34pC3ubJNcP0HWBFIr4nJhJcSshyUEMzaa6gjTb7eWtWSgDY3N6boV0mVGSFN4ig0PoZAeVmlI4WwKboA8Q0BApzMLoCSM5nbcUGUQC4FmCGObADIJ4/wT3M/ghjwuTox47o66cScLlNuztvwpaaCWNQDXtFTmEa8KwWkSDSN9xo/f8XUxhiD6/fvT0dt3v/zyMb75bKmqCEjZRjKZZAJXiHfV699tqyZtG2p3a3NA9M8Wqmyp55uhoYd5KjNsFEcNC/VRUpokZTxOSiVS2uZaBFq3mUy1eLNXCGuSpZhpTTnGwh09E+scpMweSgWmtZJwZcmGnkGKIdUYZjBI1unT+ZvLMw4yNFVtqjooZGfQVDugkNSAMqajl6fnw8RSY7RNuVoy1xPfBRRjiVp4ay+XYLjvFz7ivDXim5Y/awWmB+6E0sp1wZF6LDLAyURgLv7y4RWGJi35lE+crExMf0PGTEPQMxPjsF8+vDprgsquwmyC1uW8HcUN0i9sSBMtSix1WYPykvKEoi1qt6KtKMADcDdkX+pbCPVyIrlJSEVCaldRdLVIoCCo1CAnVyQDi1LIA+t/5U3amOG8LiedFdwS0jrL61bSKpNOzSnbqm2R/ZhDKbZX8RbHZBUEuHLEQmXTWRm9WDbVrJlxNs0Mt0PpjfWPDM3j5gQg13WFDhmbgpmYrR3yi9Uo0P5dy1JKMFmi3yeY2JcUL52vPA85zXCpBT7XB/FHlwv+C/UC2Ik+SCpEm7Oqt/FqiNioSNDTKQENiUyc0oKTK2dTYWi02zXQCGgXDEXeVMgRqJvKv6n7m9CIhwS/wYseF7BOchfASxGgUMSAv3UtTYgwq2LlNYj9eud6W4OObMD+hjf5aN66usu8tY15a3s47z7PW99l3vrGvPX9m7fe5nl3dpm3sTFvYw/nrfG8e7vMu7Mx784ezpvtmr6TXetuzLu7h/Nmu2bsZNd6G/Pu7eG8+xSCGhA4q+2yiSezSdHvGcJM3C6rUT4Yk9G4SVsXrmVFocC0cgyHaCsG3TlnVHzoiXuJlvS+YzNQwrU3KcZw/9dxQ8pdud9AuRv60LXNcJ5mIAPtYk/OCeOHVkKXE6r4oTHHDbCofeB4p5tKGsUgksGuOcVMTC6lxOEL/YozYZtZ2Tjt1BRGcRR8c6py3g4FiNNFUQzHkD9XdOsamszU9iMtGga3SUBr0uZOyxP6DSlx2y/HLEqf1GFBstvxHOO55SM4tZ3yQJc8hHXqoNh2jaYxSLiofkssfKxFzEK9Egs1bvulaBN24xzbKbyc2KnXv3w4O/6GmGZUYFqnEtMMbvsUmNatwLReJaZ1ue1TYFq/AtMGlZjW57ZPgGlaBbegVXILlJCDtk+BaRUcgVbJEWgat30KTKvgCLRKjkAzuO0XyhzFU5VAIbAiCSj49FNHHaSgTLbDPV/OsJqTwA9DPj9Hh7qSI4qm6KiaPDpGxwbxrDOfZQ3lmdOQjyze45EwPN6BG0P+THCKN7JdK1TEK3mWgs4GPuufnNC5wGe6Rt/wKMYPJ/DcJpHKn/WTJ5yK5ygV8fPZh7NzOvlXs1w7GuGWjLJc17Pnl3WtNfOdqVh5ge84eGZkGfh35tjBfNp85QBOF+fJaau/QXTIe1wDTcWzba0/iL/xAeb/LSFV11UtoYTz42MSRxfvRe1vnfYz4U8mq6XpTdb1YworxpY3WYgrPN/WWi7M0Bpfi9Ojl2JqTUAqge0LC0+4yJOiMBDPilpjy4x436rPIRwfaMfD5njx5Sc+spweZf77H9IjcRvQNn3H6BTEDX8eP/2VaVJ2+itzO3f66/eO4vXVQXr6i05LPnYW7xFCqqb1+VBfGaHdz5Gpbc3IDeyrDpJ1Bj3cw+8aenNQwvpMPphEEwZw/I8t+qe7Rd/TVDoW2+2lmprdbNz0ghrNzqA9nRxj6SD+7x2oiX1zZp9EtuPUDjSMN7NSHhj8yBI54uNKJEKiJiWWTjPSLOslYpXSyotVQgnUt4zSBuslGf7d2SDUIDsSH/gq7cnSYcTsAAcf57GAHVd455oZZhzLs8s9WqW+vpm9O23/Y6G+pYXq8CnzwaCwUOo/FupbWqguIZS+2ikslPaPhfqGFqrPf1TX73/VRnR7D7ei+5RBbwzaX7MVDeByD2fe45lrX7MZrbb3cDu6P+CZG1+zHa2293BDeqDyzLtfsyGttvdwS3rAFm6wk4Xb3JJW23u4KT1gC6e2dzJxm7vSansP96UHPTn1nWxcvzD1/h5OfSCnvpORGxSmPti/qQMOkXPfycyp7cKp2vY+Tl6Xk98NyxXAXPFM8T5MXto6dSdbpxbwnLqHeA4stJz8TtZOLUA6dQ8hHRgqOfmd7J1aQHXqHqI60FU5+d0MXgHYqXsI7EBc5eR3M3gFbKfuIbaDFePJa7sZvAK6U/cQ3cGg5eR3M3gFfKfuIb5TVWnwtN0MXgHhqfuI8DRp8LSdDJ5WQHjaPiI8TRo8bSeDpxUQnraPCE+TBk/fLWFXQHjaPiI8TRo8fSeDpxUQnraPCE+TBk/fyeBpBYSn7SPC06XB03czeAWEp+0jwtOlwdN3M3gFhKftI8LTpcEzdjN4BYSn7SPC06XBM3YzeAWEp+0jwtOlwTN2M3gFhKftI8IzpMEzdjJ4egHh6fuI8Axp8IydDJ5eQHj6PiI8o6828YBeQ+1ovaamGU/r75s2mPzVZ7I/fWOHsvfjT5q+Oa7txd80fXNc24s/avrmuLYXf9X0rXFtP/6s6Zvj2l78XdM3x7X/kz9s+n/PtQreQKvkDei1AtD2SXCtgjfQKnkDemkmtH0KXNMreAO9kjegoUPbJ8G1Ct5Ar+QN6MnQ9klwrYI30Ct5A/pjJ2j7JLhWwRvolbwBJUKg7ZPgWgVvoFfyBnqf2z4FrhkVvIFRyRsYbW77JLhWwRsYlbyBoXHbJ8G1Ct7AqOQNDIPbPgmuVfAGRiVvYHS57ZPgWgVvYFTyBkaf2z4FrnUqeINOJW/QaXPbJ8G1Ct6gU8kbdDRu+yS4VsEbdCp5g47BbZ8E1yp4g04lb9DpctsnwbUK3qBTyRt0+tz2KXCtW8EbdCt5g26b2z4JrlXwBt1K3qCrcdsnwbUK3qBbyRt0DW77JLhWwRt0K3mDbpfbPgmuVfAG3UreoNvntk+Ba70K3qBXyRv02tz2SXCtgjfoVfIGPY3bJm+alm91K3vl49SezUSrNbcjYZaVt3f9qRKEYrz93gHV2xLaTJ9oXV1RrPG02551hNpudw3jAM/2PUL5oNFoPEqdXhLYpTezd+WL2ZerseCqoOItNb6wIvEbT+qIKqxSpVLfs/ANi/h2SazobkZULEjMAt8VNz+/e/Uvr09H51qne3OMNUXT7lc3CdXhEF8RObI8c+xY05tr+bJ2LuU19n2nGVfFORKf+C2L8pWKbu6liuHCXzlTMcG3vuMLKkVaWznEetf4Cnmt9TolJssqZ4afG/bP529ea6/lwBulA8fCrNo0O3Rqx5fjwdNsZiOQ2qFkZXIttJ3VCAS+cCPwl1Z6kf+MS6NigCosEJ30tN2lU1wa/LFnxLzsJfyxloHtRY73Xe3wikXhmtthXTQb+IVvoBRyLqIGd2iF4xr2+FLS+mH9OCX6hWfLnJGVbbE08tSempGFFd3sMFO6zQzsaOFakT2JVwzZj7Wqplh5MkcssJaOiaUct5bI5cXjEnD3QDgpkvuG3v6/DnP0/GXUAhFYeSAwXMg98Key4u3c4Xd7zrGkHDyY7lhYoVtcGkpKxgEu88qKE+G6pmKHo9B3rVpdPH8Oz5wOh5Z3NxzemcHID2uHOTE6rKfNj1OasFSS5G/pxW2LBcIr6/xSRfKyZTvMEv+SLtUvt7VSUSGD65rN/JWA3vqaGw+PcqOdFGowIivHArcQjWawKMjU2uHcwZsjMJuH9X8u9EsFf1vnuMU2Cqwh23rj3aQnvYezQxWdO91eU9UeVx8sfK8gCw5iLqbaf8E2JmZ9uBq7aNSdXOXmrLVBfyBl9Z/B0Hxe2UFaX5wLOIfuqNeRM6GCzRsGCKlRrWTvzg58D9/5KaUSbfTME3krVHuOE6jjq23R/mQFi2bGrfnil7iQLI7lpYm1EacsTzd/ucL3+Tbx1cDXIO9/lv+0vWvxo/h0BZfpH/9+eRO/zfbNh8t+Si1XVrH2XlNUcWmGt+JlkyfcqCviwnQtUE0rgFmHwgyz7uECxgoGFsvJf+6P2qA65s21+O///C96FrC65Zq/wgP+BDfFhX8qPoeCX4FMJagdfM/mmpceYCH4tEanH/u28rWnUaN6hdk3JpNfwOLJfVHDJzuOCao/WS7FAlq0Ir+Fn+h87q1pSiixmvK9zFg47F4Wi4+lZ2LZDlyvK0k3xibYPxYVcQOcHk1tVzyDEZyciPZNU9zYnrwEoCS+RtW9fzgBQbyhkaa0gpXnwQToTdEhuuibj+dnP71592708vTy1R9v6k0WO3zF883RTfyS55v0Lc8pLSzjSY+64dc+g7zwC6rzRb+BWdZDBCCQXsNrBha997QpPKwZmFLD4iwgKCAKZ0HgAwQCe4gcksqAb1H2o7j8IK492WjfSxVoZz4lngZ7PzLlyjNNqT0y5cxMT8kssAU3g8hGm4XtxzZ0TouyH8UPpwrwCP+WloelTmNL8+ryNNyZbwUFW5jhCBxAgrr+6Qo5cF+bOPZyuR4OI98fuaa3HpnBfIWmJ6xfc8vY+KCGAgXQ0hqrW7+Pf5LSbcuCM+XKRvbp3AqBkz/UQAB/dmgmf8ga4qk1Xs1HZgiOPxpZn7+rZZa4KQDyH2YeDtyWkrApCDnMUqCZEZFHaHKJTV1rvc1VVUfSrTLS39VSddygSfiU1NAFXANjZBQLRhJCmvTd1fWcKy+h/Yfto0UUbplAHJ1QQjLHBwQztVFTzJpiVAf7jq4h72IVoLpxBWQFTFeNq1N32x0sLNTVdFm8dLtHxZ/n7ioSNhh58QK/jZJvk9Gdb28iC2rtVWl9nVmB+AXgC9S8EGbV5yqo9E0Wf2dek4kiRUXTLis0JnQQbinstGsvgEM1tNQgJnexBGKpdhBaiElV+EConPwDFoUGT28FD+s5FPoVg5PwdjtIlugYsTKBEELGx2TFUlEUAFbAOFgP5iRy1kLNoNvsnPPQDziQv7CNHcllhgrMovpm5wyz8nfaG//OsDFzp56iMsmHI/ER38cvXkqn26JQhWDMMFMdeiKrHaAJz8EL1p6b6/iVYAMEimrbMJodFeUajCqgmTAKsyKdsSAfL/88unjf6yguAaiw9v1v39eVCfiMCC1b8faX9HZWJWPtTjrgiE2AIrVDJQLzCzpFyOmwai+MUbDcGb/k3poqbt/r36pdBWQLmjmYilHCPv2HUDlrcNKALhNxBNbYDBHJYLFfb44hEkhR5JeF40peVFH0JSiLLWBc0gI9HHQBv7MmHOKuQg7hwNOq4CplmJ4PxGTKIArWoD/x3PPyo8zAawLv7uzQBmAsFG4O0UFm2ak8Re2wvtHVelhak6h2SOOmfjlzjCPA1/CPZCB7Ip7HY7hSlHRs18cloy7pk/ZQlGyfjJRlHpdI0uGzCSw1mIXDjLyp9XICmWc/TqBdJJB/fEbCbC/O52ECbjRfg4Mgc1CQJqriJ3A1wSctTTsgmD79FQwYYBpc936L3lYshZUKnYx9COxlripHjWQksHAY5OTuYLXA0oURVgFEqyllEel6PvlsM+DHUptQKeVRQVcPN3OAWUZhuQVdK+PWdyXaWJqTZDblpnYqDMmJU8J5VLUCmUXMAHozx55ErVlgWWTdUfc4yAQ+uOYyR4zmb0argLWLWZt0k4BhipAS4Ccq3ZwqHDOLxOXCylGLLHcZYQO9yyPksYVmZIcz28KaJzal55YYfwfRusDlUjtFZTz0di9mR9UuXPmjrM/2xUzfbp3TGVhJY1PB0f2MTQyZTsS/WpPhENNMo4m5NCd2tK7lVx5ZSoUwRmAnQSjbitLfzOkQ2+25vG1s3sYfep6yXIWLWq2W0KOXgVOxDPxaF0f469mG7OFPJiOX+cpEIRqPRisIbDFLkEtC8X0IFVbL2hZechPH8pBbiJRpBRKZY32M/BVIA1eFxw4UOx9ucgllmjnAOcoCi1CyRvzOcmx4hdsPsESAGjTjuoxpmXHWknenN7J06jJkeJxfOVl/mTE9lMQEb+SZLijLRkV20rFlXK79dtxQZTXSvAP0I5NUBxGYJ11mXK8d1TdNg0Sc7XmwQ+owNoPAtoKqBuvRYj5ZmS8xXEQyz6Fy14o/6SN/vzL8pn+l/vFI8re0DNQrgTkllvX7f2t/XwqJStsG38PcD1+dv/tJ3FMub4x28FfwH7C047VYRg9mmAuVILZGc3CBuVvH9qx6BnEoeCGs1QFbQHQcQJQN3wl3/BWjK6f+V/Gdg5lfM5zYNuA9kN9tu0GcpilsBCWX5R7QeDzQB6qhKO3+ZGBos617QGnHwvZPeouLy9NLJPgDLnBUfze6H4FY1w4IYlMu3+ZQg4Nhn3cZvkcvfmd6WHCXfDaHGLl2+A+skxXavqcwvU+cH4uVKE2YpAk3jGKasmqYIGHhVjcYZ9S8I4gvbiidw9kSeRVrAN0cg7JGNpgmy/RCBp9gCBB0eqDWqFWotI4fcppflllwLdcP1lig7EjUqNQYVaAS5hxlJxKa3saQ5n5hT9ARY0TAaQAa7sV7BUuSQVcuLpbvq+ta//G+lP+jqby2HHtMwgQ+G9BLUooMHHDgKuLUQwxtzWb2xEbzwe4c7EYi0LZ3BzcsyRqgxHtmpoccCVcw2WMUazO7DYLFzDCLKfkFzCEoAFJ/TyWWzTHAdpaXgYYhEn4AjxKBcUeI5mv5QJ5QA+L5hbm0wjphijR9PglMCW5yGcmk0l2RFoMdEDqUlRYW9sJRgj3tZ+t+UQb43lxTJp5KrIVKnhbM/laRyS/ewMGOqKlxcg9cR61fL/M3JZtdQ0Khck8KggeSLPaEgK3SjchSYtECHP083cFCFskggSIuIPVwxAnKozXJy6vL0/yWWYYaMBfM351MC/iBDZAOwl9cTcdqUWoA/c3Yh0gMQF8LpAfBn1JK7C06PipnZy5kQpz3ZZF2dO+TQpoOxW9Um5uYHvnlIzMRhoJw/4d1xDE6USHmmA/4jDj6AxEGcfx+CqRxg7lIjZZvY9+jdK3wB+KEVeBhh2yusrQp/qCVBE39DBJ4H06aVPkQf/MKNMW6KeXT9sA+ec1ySptQA3++HBSaxW/m+USGARcG8xackAfZvUe1pjULgWuZt/bkaZTS5dgVYQSuiJj6Sd6H0oJoBodsci0Tns3JE67S3tYxtaer+PG4fj/y/Evz1pK2F9MEYEU8QUFGYNMowAqtOHtlRmDUMN9HSapSYiApazLpwCsKhGjg5GW43iX7DfSBwDm2/nPfIllsldOLE+msarFtXqG6oaWXBhwkfsM9lFCLrbx0iOQ7cBxod7ky/cyGIbr4bdNl7Dw4dCXbBldGbWNw7J0qjq50XWF0M9O1HRtC6kPMZ4LbcMg2QFQFtxyHkvCx50/3/UvJZfbaxhYZ8/XSQuEvfbosSNND+ez3e021n6Sef16u3vtTy9nMPGP1vYi2dMyQ+Raz0w5ZwXDPxp8hlEOLjsUeVxQv/7qazq2i47gHtAPQyJ9hhuG34igRLXoeoERKYK/wVFLZurzz55SYTLYKMS1M5m/m4x6MZJ7Lvj0ZFEQJ5VJDPkkDd+tYwO1gHbvAyQKTh9PcDrVEauzry6UGHkx5OmYdbQqz0AFINh4wTPGoPKwSn2CAcAV8OvjUUnrkH+WMSXdRxOSQ1i10aDVA6XFE9E4TC1viPbOU3gRsFkRJgOomUZO2u2bWPZCUjhP3vsL6Mdsf1mmZbixhHi4YsoUMHKwbrPF3Je4FQMKtkj2rJIEDXlkCp2uet9UHlbL5i4Dg0trWBSxhscdxubhhrgKwEIz9r2E0hAmDt2gKdyieo1aY0V/hIQXDhT+Y8rFGtLBXYXQtGidSHvla7bmr3NchopYXSUjpIvDcoxx/iXD/HQgny1EvmXFK3zUnCXmP1a1rAFVXiXcBM5fkVh9fKSFbxlzE1o65BsECb0znJ+hfgBQB89S2hXO0XXC0RMu3zgV0GzdkSNfpTLpdE0K6qdHt9sblx/o2u+aCus2bdKDPUOlEH3xoOZguM98yfh9Nrc8rsGggMVPc3gOfAJ6UpsZcOeAjMEcJ7B2WHr/z0LYsMd7KnrvLY9vY2DM1HyOQ7L4VblkR/A3REKPScgnsjD9EQ0GA3gzWiMAmzmpqMTkIJdHMQaDvhAyglvLkjOd7raUP1qHlz1qIWkEMsLZX7hSY5ZPVBPPwT1fIzuuDxgbDpEH4fb41YrWknEGNAeVtHRMH8+UKmrDax8D0i1QhNjtpUJKxC5njYBdv33wcQjh9h6W9cSPmXp4gwn1+iBb54E0LD97I8wC5FBw/8zg5/xMLebqXx2rCBcDrlIFLe9fUrrQtXSP+0qEv2TxODZN1uPGttdvl1yFIz16nPUEstpbtkcv1JeuAclF7zix9zueTisNuIgoJrXo6TTrpdCTRCwejHNdY3hz9fupeVx5ID3LkT/eWpymdVlvpvBQ1tBw4QqZCW1auNbXxXIXRBzMoA3hSvE67qRqgeT341P93mocPMR06IZCkLMYIqjIiO7YW5p3trwJaelPMzSUMHOuJp4Cb0vx0lg0pSlWdgYrwoUu6j/9cgbSgYMBlfArECxCDrTw8Dgna6EUc3cn0PsAKOd1WzFLCjdksDR108oFPHM/S/iGw+6Z07/iGj3B6PhMMlw5YUzrH+1ZgcoNOZRFmiY/53HBomJRfX4AauJh+TtJNqB9MDh4LCIAPwvS6McSl8z14Lubm6Ga1vGliiRe6NvXvPbjmw/+f4bLGV2/hn3c3TaYIzDeF0W5dvMcjouI0SaskqY0YHuKuPlNsAEVETd3uMxykbMj0AI5CWx5Ng0fj+ncEhu9RzdM4Jwawke/fgpt4JnNOyOKHFgXRuZUILBe3yAAf+0EEojZEBmyZOVPanD4KSY4Fm5OXqRCZNrFJr/xQ2mTAijfFYwE3G27hGCcIFhEbnICzZts/zaTBmBql4bnLiZqz8EmaQ6ZgwMebIE4tf5l4i6zExkerwMGYS4w2cF0g4rMmJqclTEpi3mTPB0DoFB8hOGa1oiQqE5Rn63BXnLe/3GUUKpQ+kUl/mdaH50EYbgUTG6eIB0d5WeUhILmccT86uWHyOZ7ZChqj+3uk+7E8CoKZlkgugUTnlIWTO+/kVVsyiST9Zgv9ZmxSAmAWTBZ6nJ+dvhtd/PH049nFUFzVpM3PfgDINvCE6BXDhRqZcfpFB0AIrH+mB/kYNP06RHFjHQSNBgMYyD8LqKn43q+NnrfU824IMij1Nj4C6pkBAAKclTw8I0XBxRjw4H8AhXbBPOewAAA="""

# Official Qwen GGUF, pinned so every A/B session reads identical bytes.
NOTEBOOK_BUILD = "wave3-grid2d-v2-parser-fix"
HF_REPO = "Qwen/Qwen2.5-0.5B-Instruct-GGUF"
HF_REVISION = "9217f5db79a29953eb74d5343926648285ec7e67"
HF_FILENAME = "qwen2.5-0.5b-instruct-q4_k_m.gguf"
HF_EXPECTED_BYTES = 491_400_032
HF_EXPECTED_SHA256 = "74a4da8c9fdbcd15bd1f6d01d621410d31c6fc00986f5eb687824e7b93d7a9db"
MODEL_PATH = ""  # populated by the pinned fetch cell

# Decision-grade defaults. Keep these fixed between reruns.
TARGET_PREFILL_TPS = 15_000.0
T4_INT8_TMAC_S = 65.0  # 130 TOPS when multiply and add are counted separately
FORCE_Q8_FOR_PTX = True
MICRO_REPEATS = 3
PRODUCTION_REPEATS = 2
COLD_ITERS = 3
WARMUP_ITERS = 1
MEASURE_ITERS = 5
INCLUDE_R256 = False  # explicit r256-vs-grid2d arms below
RUN_NCU = True

WORK = Path("/kaggle/working") if Path("/kaggle/working").is_dir() else Path("/content")
RUN_ID = dt.datetime.now(dt.timezone.utc).strftime("%Y%m%dT%H%M%SZ")
ROOT = WORK / f"glcuda-ceiling-wave3-{RUN_ID}"
META_REPO = ROOT / "meta"
BASE_DIR = ROOT / "base"
CAND_DIR = ROOT / "candidate"
RESULTS = ROOT / "results"
BASE_TARGET = ROOT / "target-base"
CAND_TARGET = ROOT / "target-candidate"
for path in (ROOT, RESULTS):
    path.mkdir(parents=True, exist_ok=False)

def run(cmd, cwd=None, env=None, timeout=7200, check=True):
    merged = dict(os.environ)
    if env:
        merged.update({str(k): str(v) for k, v in env.items()})
    proc = subprocess.run(
        [str(x) for x in cmd],
        cwd=str(cwd) if cwd else None,
        env=merged,
        capture_output=True,
        text=True,
        errors="replace",
        timeout=timeout,
        stdin=subprocess.DEVNULL,
    )
    if check and proc.returncode:
        tail = (proc.stdout + "\n" + proc.stderr)[-5000:]
        raise RuntimeError(f"command failed ({proc.returncode}): {' '.join(map(str, cmd))}\n{tail}")
    return proc

def save_log(name, proc):
    path = RESULTS / name
    path.write_text(
        f"$ {' '.join(map(str, proc.args))}\nexit={proc.returncode}\n\n"
        f"--- stdout ---\n{proc.stdout}\n--- stderr ---\n{proc.stderr}",
        encoding="utf-8",
    )
    return path

gpu_proc = run([
    "nvidia-smi",
    "--query-gpu=index,name,compute_cap,memory.total,driver_version",
    "--format=csv,noheader,nounits",
], timeout=60)
gpu_rows = [line.strip() for line in gpu_proc.stdout.splitlines() if line.strip()]
if not gpu_rows:
    raise SystemExit("No NVIDIA GPU is visible. Enable a Kaggle GPU accelerator.")
print("Visible GPUs:")
for row in gpu_rows:
    print(" ", row)

first = [x.strip() for x in gpu_rows[0].split(",")]
if len(first) < 5 or "T4" not in first[1] or first[2] != "7.5":
    raise SystemExit(f"GPU 0 must be NVIDIA T4 compute capability 7.5; got: {gpu_rows[0]}")
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
print("Pinned CUDA_VISIBLE_DEVICES=0")

if shutil.which("cargo") is None:
    installer = run([
        "bash", "-lc",
        "curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | "
        "sh -s -- -y --profile minimal",
    ], timeout=1200)
    save_log("rustup-install.log", installer)
os.environ["PATH"] = str(Path.home() / ".cargo" / "bin") + os.pathsep + os.environ["PATH"]
if shutil.which("cargo") is None:
    raise SystemExit("Rust installation did not expose cargo.")

clone = run(["git", "clone", "--filter=blob:none", "--no-checkout", REPO_URL, META_REPO], timeout=1800)
save_log("git-clone.log", clone)
have_rev = run(["git", "cat-file", "-e", f"{BASE_REV}^{{commit}}"], cwd=META_REPO, check=False)
if have_rev.returncode:
    fetch = run(["git", "fetch", "--depth", "1", "origin", BASE_REV], cwd=META_REPO, timeout=1800)
    save_log("git-fetch-base.log", fetch)

run(["git", "worktree", "add", "--detach", BASE_DIR, BASE_REV], cwd=META_REPO)
run(["git", "worktree", "add", "--detach", CAND_DIR, BASE_REV], cwd=META_REPO)
actual_base = run(["git", "rev-parse", "HEAD"], cwd=BASE_DIR).stdout.strip()
if actual_base != BASE_REV:
    raise SystemExit(f"Baseline checkout mismatch: expected {BASE_REV}, got {actual_base}")

patch_bytes = gzip.decompress(base64.b64decode(PATCH_GZIP_B64, validate=True))
actual_patch_sha = hashlib.sha256(patch_bytes).hexdigest()
if actual_patch_sha != PATCH_SHA256:
    raise SystemExit(f"Embedded patch digest mismatch: {actual_patch_sha}")
patch_path = RESULTS / "candidate.patch"
patch_path.write_bytes(patch_bytes)
run(["git", "apply", "--check", patch_path], cwd=CAND_DIR)
run(["git", "apply", "--whitespace=nowarn", patch_path], cwd=CAND_DIR)

base_status = run(["git", "status", "--porcelain"], cwd=BASE_DIR).stdout.strip()
cand_status = run(["git", "status", "--porcelain"], cwd=CAND_DIR).stdout.strip()
if base_status:
    raise SystemExit(f"Baseline tree is dirty:\n{base_status}")
if not cand_status:
    raise SystemExit("Candidate tree is unexpectedly identical to baseline.")

ptx_path = CAND_DIR / "glcuda/src/kernels/glcuda_sm75.ptx"
ptx = ptx_path.read_text(encoding="ascii")
markers = {
    "vector_store_pairs": ptx.count("st.global.v2.f32"),
    "scalar_f32_stores": ptx.count("st.global.f32"),
    "u64_shared_stages": ptx.count("st.shared.u64"),
    "mma_instructions": ptx.count("mma.sync.aligned.m8n8k16"),
    "sm_a_3072": ptx.count("sm_a[3072]"),
    "sm_a_12288": ptx.count("sm_a[12288]"),
    "grid2d_rebase": ptx.count("mov.u32 %r_gy_t0, %ctaid.y;"),
}
expected = {
    "vector_store_pairs": 40,
    "scalar_f32_stores": 0,
    "u64_shared_stages": 5,
    "mma_instructions": 80,
    "sm_a_3072": 1,
    "sm_a_12288": 1,
    "grid2d_rebase": 1,
}
if markers != expected:
    raise SystemExit(f"Candidate structural markers changed:\nexpected={expected}\nactual={markers}")

print(f"Baseline  {actual_base}")
print(f"Patch     {actual_patch_sha}")
print(f"Run root  {ROOT}")
print("Candidate structural markers:", markers)


## 2 · Fetch the pinned production model

This cell downloads the official Qwen2.5-0.5B-Instruct Q4_K_M GGUF directly. The repository revision, expected size, and LFS SHA-256 are pinned in the configuration cell. A valid cached copy is reused; a partial download is resumed. The model fetch is outside glbench so the benchmark remains observation-only.


In [ ]:
print(f"MODEL FETCH START [{NOTEBOOK_BUILD}]")
sys.stdout.flush()

import urllib.error
import urllib.request

MODEL_CACHE = WORK / "models"
MODEL_CACHE.mkdir(parents=True, exist_ok=True)
model_dest = MODEL_CACHE / HF_FILENAME
model_part = model_dest.with_name(model_dest.name + ".part")
model_url = f"https://huggingface.co/{HF_REPO}/resolve/{HF_REVISION}/{HF_FILENAME}?download=true"

def file_sha256(path, chunk_bytes=8 * 1024 * 1024):
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        while True:
            block = handle.read(chunk_bytes)
            if not block:
                break
            digest.update(block)
    return digest.hexdigest()

def park_invalid(path, reason):
    parked = path.with_name(path.name + f".invalid-{reason}-{RUN_ID}")
    path.replace(parked)
    print(f"Parked invalid cache file: {parked}")

def validate_model(path):
    if not path.is_file():
        return False, "missing"
    size = path.stat().st_size
    if size != HF_EXPECTED_BYTES:
        return False, f"size-{size}"
    with path.open("rb") as handle:
        if handle.read(4) != b"GGUF":
            return False, "magic"
    digest = file_sha256(path)
    if digest != HF_EXPECTED_SHA256:
        return False, f"sha256-{digest[:12]}"
    return True, digest

valid, detail = validate_model(model_dest)
if valid:
    print(f"Reusing verified model: {model_dest}")
else:
    if model_dest.exists():
        park_invalid(model_dest, detail)
    if model_part.exists() and model_part.stat().st_size > HF_EXPECTED_BYTES:
        park_invalid(model_part, f"oversize-{model_part.stat().st_size}")
    if model_part.exists() and model_part.stat().st_size == HF_EXPECTED_BYTES:
        partial_valid, partial_detail = validate_model(model_part)
        if partial_valid:
            model_part.replace(model_dest)
        else:
            park_invalid(model_part, partial_detail)
    if not model_dest.exists():
        start = model_part.stat().st_size if model_part.exists() else 0
        headers = {"User-Agent": "GwenLand-Wave2-Kaggle/1.0"}
        if start:
            headers["Range"] = f"bytes={start}-"
            print(f"Resuming model download at {start / 2**20:.1f} MiB")
        else:
            print(f"Downloading {HF_REPO}@{HF_REVISION[:12]}/{HF_FILENAME}")
        request = urllib.request.Request(model_url, headers=headers)
        try:
            response = urllib.request.urlopen(request, timeout=120)
        except urllib.error.HTTPError as exc:
            raise SystemExit(f"Model download HTTP {exc.code}: {exc.reason}. Confirm Kaggle Internet is On.") from exc
        except urllib.error.URLError as exc:
            raise SystemExit(f"Model download connection failed: {exc.reason}. Confirm Kaggle Internet is On.") from exc
        status = getattr(response, "status", response.getcode())
        if start and status != 206:
            print(f"Server ignored Range (HTTP {status}); restarting the partial download.")
            start = 0
        mode = "ab" if start and status == 206 else "wb"
        downloaded = start
        last_print = time.monotonic()
        with response, model_part.open(mode) as output:
            while True:
                block = response.read(8 * 1024 * 1024)
                if not block:
                    break
                output.write(block)
                downloaded += len(block)
                now = time.monotonic()
                if now - last_print >= 5:
                    pct = 100.0 * downloaded / HF_EXPECTED_BYTES
                    print(f"  {downloaded / 2**20:.1f} / {HF_EXPECTED_BYTES / 2**20:.1f} MiB ({pct:.1f}%)")
                    last_print = now
        part_valid, part_detail = validate_model(model_part)
        if not part_valid:
            park_invalid(model_part, part_detail)
            raise SystemExit(f"Downloaded GGUF failed integrity validation: {part_detail}")
        model_part.replace(model_dest)
valid, digest = validate_model(model_dest)
if not valid:
    raise SystemExit(f"Final GGUF validation failed: {digest}")
MODEL_PATH = str(model_dest)
MODEL_FETCH = {
    "repo": HF_REPO,
    "revision": HF_REVISION,
    "filename": HF_FILENAME,
    "url": model_url,
    "bytes": model_dest.stat().st_size,
    "sha256": digest,
    "path": MODEL_PATH,
}
(RESULTS / "model-fetch.json").write_text(json.dumps(MODEL_FETCH, indent=2), encoding="utf-8")
print(json.dumps(MODEL_FETCH, indent=2))


## 3 · PTX assembly and resource report

Both source trees are assembled directly with ptxas for sm_75 before Rust builds begin. The raw verbose output is archived; register count, stack usage, spills, and static shared memory must be read from those logs rather than inferred from PTX declarations.


In [ ]:
ptxas = shutil.which("ptxas")
if ptxas is None:
    candidates = sorted(Path("/usr/local").glob("cuda*/bin/ptxas"), reverse=True)
    ptxas = str(candidates[0]) if candidates else None
if ptxas is None:
    raise SystemExit("ptxas is required for Wave 3 but was not found in the Kaggle image.")

tool_versions = {}
for name, cmd in {
    "nvidia_smi": ["nvidia-smi"],
    "rustc": ["rustc", "--version", "--verbose"],
    "cargo": ["cargo", "--version"],
    "ptxas": [ptxas, "--version"],
}.items():
    p = run(cmd, check=False, timeout=120)
    tool_versions[name] = (p.stdout + p.stderr).strip()
    print(f"--- {name} ---\n{tool_versions[name][:1500]}")

def parse_ptxas_function_resources(text, fn):
    # Exact, line-anchored match is required: gl_gemm_mma_q8 is a prefix of
    # gl_gemm_mma_q8_r256, which ptxas commonly prints first.
    header = re.search(
        rf"(?m)^ptxas info\s*: Function properties for {re.escape(fn)}\s*$",
        text,
    )
    if not header:
        raise ValueError(f"missing Function properties header for {fn}")
    tail = text[header.end():]
    next_function = re.search(
        r"(?m)^ptxas info\s*: Compiling entry function\b",
        tail,
    )
    block = tail[:next_function.start()] if next_function else tail
    match = re.search(
        r"Used\s+(\d+)\s+registers.*?(\d+)\s+bytes smem",
        block,
        re.S,
    )
    if not match:
        raise ValueError(f"missing register/smem counters for {fn}")
    return {"registers": int(match.group(1)), "smem_bytes": int(match.group(2))}


# Regression fixture deliberately lists the longer, prefix-sharing name first.
_PTXAS_PARSE_FIXTURE = """ptxas info    : Compiling entry function 'gl_gemm_mma_q8_r256'
ptxas info    : Function properties for gl_gemm_mma_q8_r256
    0 bytes stack frame, 0 bytes spill stores, 0 bytes spill loads
ptxas info    : Used 110 registers, used 1 barriers, 13312 bytes smem, 388 bytes cmem[0]
ptxas info    : Compiling entry function 'gl_gemm_mma_q8'
ptxas info    : Function properties for gl_gemm_mma_q8
    0 bytes stack frame, 0 bytes spill stores, 0 bytes spill loads
ptxas info    : Used 52 registers, used 1 barriers, 3328 bytes smem, 388 bytes cmem[0]
"""
_expected_parse = {
    "gl_gemm_mma_q8": {"registers": 52, "smem_bytes": 3328},
    "gl_gemm_mma_q8_r256": {"registers": 110, "smem_bytes": 13312},
}
for _fn, _expected in _expected_parse.items():
    _actual = parse_ptxas_function_resources(_PTXAS_PARSE_FIXTURE, _fn)
    if _actual != _expected:
        raise RuntimeError(f"ptxas parser self-test failed for {_fn}: {_actual} != {_expected}")


def assemble(label, src):
    cubin = RESULTS / f"{label}-glcuda-sm75.cubin"
    ptx_file = src / "glcuda/src/kernels/glcuda_sm75.ptx"
    p = run([ptxas, "-arch=sm_75", "-v", ptx_file, "-o", cubin], cwd=src, timeout=600, check=False)
    save_log(f"ptxas-{label}.log", p)
    text = p.stdout + "\n" + p.stderr
    print(f"\n--- ptxas {label} ---\n{text}")
    if p.returncode:
        raise SystemExit(f"ptxas failed for {label}; see {RESULTS / ('ptxas-' + label + '.log')}")
    if "spill stores" not in text or "spill loads" not in text:
        raise SystemExit("ptxas did not print explicit spill counters; resource gate is unverifiable.")
    spill_values = [int(x) for x in re.findall(r"(\d+) bytes spill (?:stores|loads)", text)]
    if not spill_values or any(spill_values):
        raise SystemExit(f"ptxas spill gate failed for {label}: {spill_values}")
    resources = {}
    for fn in ("gl_gemm_mma_q8", "gl_gemm_mma_q8_r256"):
        try:
            resources[fn] = parse_ptxas_function_resources(text, fn)
        except ValueError as exc:
            raise SystemExit(f"Could not parse ptxas resources for {fn} ({label}): {exc}") from exc
    # 256 threads/block on T4 retains four register-limited blocks through
    # 64 registers/thread. Crossing this line would invalidate the grid A/B.
    if label == "candidate" and resources["gl_gemm_mma_q8"]["registers"] > 64:
        raise SystemExit(f"Wave 3 occupancy gate failed: {resources}")
    return {"cubin": str(cubin), "log": text, "resources": resources}

PTXAS = {
    "base": assemble("base", BASE_DIR),
    "candidate": assemble("candidate", CAND_DIR),
}
PTXAS_OK = True


## 4 · Correctness gate on the real T4

This gate runs both baseline and candidate GPU suites serially. A green Cargo result containing SKIP: no CUDA driver/device is treated as a failure. Timing cells must not run unless every suite really executed on the device.


In [ ]:
def cargo_env(target):
    return {
        "CARGO_TARGET_DIR": str(target),
        "RUST_BACKTRACE": "1",
        "CUDA_VISIBLE_DEVICES": "0",
    }

def cargo_run(label, src, target, args, log_name, timeout=7200):
    p = run(["cargo", *args], cwd=src, env=cargo_env(target), timeout=timeout, check=False)
    save_log(log_name, p)
    hay = p.stdout + "\n" + p.stderr
    if p.returncode:
        raise SystemExit(f"{label} failed (exit {p.returncode}); see {RESULTS / log_name}\n{hay[-3000:]}")
    return hay

TEST_RESULTS = {}
for label, src, target in [
    ("base", BASE_DIR, BASE_TARGET),
    ("candidate", CAND_DIR, CAND_TARGET),
]:
    print(f"\n=== {label}: host library tests ===")
    host = cargo_run(
        label, src, target,
        ["test", "--locked", "-p", "glcuda", "--release", "--lib", "--", "--nocapture"],
        f"test-{label}-lib.log",
    )
    TEST_RESULTS[f"{label}_lib"] = host

    for suite in ("parity", "forward", "graph_replay"):
        print(f"=== {label}: {suite} ===")
        hay = cargo_run(
            f"{label}/{suite}", src, target,
            ["test", "--locked", "-p", "glcuda", "--release", "--test", suite,
             "--", "--test-threads=1", "--nocapture"],
            f"test-{label}-{suite}.log",
        )
        if "SKIP: no CUDA driver/device" in hay:
            raise SystemExit(f"{label}/{suite} silently skipped CUDA device tests.")
        matches = re.findall(r"test result: (ok|FAILED)\. (\d+) passed; (\d+) failed", hay)
        if not matches or any(state != "ok" or int(failed) != 0 for state, _, failed in matches):
            raise SystemExit(f"Could not prove {label}/{suite} passed on hardware.")
        TEST_RESULTS[f"{label}_{suite}"] = {
            "summaries": matches,
            "skip_count": hay.count("SKIP: no CUDA driver/device"),
        }
        print(" ", matches[-1])

for label, src, target in [
    ("base", BASE_DIR, BASE_TARGET),
    ("candidate", CAND_DIR, CAND_TARGET),
]:
    cargo_run(
        f"{label}/bench build", src, target,
        ["build", "--locked", "--release", "-p", "glcuda", "--example", "bench"],
        f"build-{label}-bench.log",
    )
    cargo_run(
        f"{label}/glbench build", src, target,
        ["build", "--locked", "--release", "-p", "glbench"],
        f"build-{label}-glbench.log",
    )

BINS = {
    "base": {
        "bench": BASE_TARGET / "release/examples/bench",
        "glbench": BASE_TARGET / "release/glbench",
    },
    "candidate": {
        "bench": CAND_TARGET / "release/examples/bench",
        "glbench": CAND_TARGET / "release/glbench",
    },
}
for arm, bins in BINS.items():
    for kind, path in bins.items():
        if not path.exists():
            raise SystemExit(f"Missing {arm} {kind} binary: {path}")

CORRECTNESS_OK = True
print("\nCorrectness gate passed for clean baseline and patched candidate on the real T4.")


## 5 · Interleaved diagnostic GEMM A/B

The repository's glcuda bench is diagnostic evidence only. Each executable contains its own PTX at compile time. Runs are interleaved and the full raw output is archived. The extracted gemm-phaseb rows compare the 64-row and r256 paths on fixed 512-row work.


In [ ]:
if not globals().get("CORRECTNESS_OK"):
    raise SystemExit("Correctness gate did not pass.")

MICRO = {"base": [], "candidate": []}
for repeat in range(MICRO_REPEATS):
    order = ("base", "candidate") if repeat % 2 == 0 else ("candidate", "base")
    for arm in order:
        t0 = time.time()
        p = run([BINS[arm]["bench"]], cwd=BASE_DIR if arm == "base" else CAND_DIR,
                env={"CUDA_VISIBLE_DEVICES": "0"}, timeout=7200, check=False)
        save_log(f"micro-{repeat}-{arm}.log", p)
        hay = p.stdout + "\n" + p.stderr
        if p.returncode:
            raise SystemExit(f"Microbench {arm} repeat {repeat} failed; see archived log.")
        phase = [line for line in hay.splitlines() if line.startswith("[gemm-phaseb ") and "512-row" in line]
        reuse = [line for line in hay.splitlines() if line.startswith("[gemm-reuse ") and " in=" in line]
        if len(phase) != 2:
            raise SystemExit(f"Expected two gemm-phaseb rows, found {len(phase)} in {arm} repeat {repeat}.")
        MICRO[arm].append({"repeat": repeat, "phaseb": phase, "reuse": reuse})
        print(f"{arm:9s} repeat {repeat} ({time.time()-t0:.1f}s)")
        for line in phase:
            print(" ", line)

phase_re = re.compile(
    r"\[gemm-phaseb\s+(.+?)\]\s+512-row chunk: "
    r"8x64\s+([0-9.]+)us\s+\|\s+4x128\s+([0-9.]+)us.*?\|\s+"
    r"2x256\s+([0-9.]+)us"
)
MICRO_TABLE = []
for arm, runs in MICRO.items():
    by_label = {}
    for rec in runs:
        for line in rec["phaseb"]:
            m = phase_re.search(line)
            if not m:
                raise SystemExit(f"Could not parse gemm-phaseb row: {line}")
            label = m.group(1).strip()
            by_label.setdefault(label, {"t64": [], "t128": [], "t256": []})
            for key, value in zip(("t64", "t128", "t256"), m.groups()[1:]):
                by_label[label][key].append(float(value))
    for label, values in by_label.items():
        MICRO_TABLE.append({
            "arm": arm,
            "shape": label,
            **{key: statistics.median(v) for key, v in values.items()},
        })

print("\nDiagnostic medians (microseconds per 512-row chunk):")
for row in MICRO_TABLE:
    print(row)


## 6 · Production glbench A/B

Production measurement is the decision authority. The preceding cell provides one revision-pinned, SHA-256-verified GGUF through MODEL_PATH. With FORCE_Q8_FOR_PTX enabled, both baseline and candidate stage the same Q4/K-quant weights as Q8_0 so the comparison measures these MMA kernels instead of silently falling back to per-token GEMV.

Four arms are available:

- clean baseline, forced-Q8, default 64-row path
- patched candidate, forced-Q8, default 64-row path
- clean baseline, forced-Q8 with GLCUDA_R256=1
- patched candidate, forced-Q8 with GLCUDA_R256=1

The r256 arms can be disabled in the configuration cell. Run order rotates between repeats to reduce monotonic thermal drift.


In [ ]:
if not globals().get("CORRECTNESS_OK"):
    raise SystemExit("Correctness gate did not pass.")

if MODEL_PATH:
    model = Path(MODEL_PATH)
    candidates = [model]
else:
    candidates = sorted(Path("/kaggle/input").rglob("*.gguf")) if Path("/kaggle/input").is_dir() else []

if len(candidates) != 1:
    print(f"Production gate pending: expected exactly one GGUF, found {len(candidates)}.")
    for path in candidates[:20]:
        print(" ", path)
    print("Set MODEL_PATH in cell 1 and rerun this cell through the artifact cell.")
    PROD_OK = False
    PROD_RECORDS = []
    PROD_SUMMARY = []
else:
    model = candidates[0]
    if not model.is_file() or model.stat().st_size < 10_000_000:
        raise SystemExit(f"Model path is not a plausible GGUF: {model}")
    if FORCE_Q8_FOR_PTX:
        print("Production policy: GLCUDA_FORCE_Q8=1 for both A/B builds.")

    prompt_unit = (
        "Measure this deterministic systems prompt carefully. Explain how token-parallel "
        "integer matrix multiplication uses shared memory, Tensor Cores, and fixed launch geometry. "
    )
    FIXED_PROMPT = prompt_unit * 8
    force_env = {"GLCUDA_FORCE_Q8": "1"} if FORCE_Q8_FOR_PTX else {}
    arms = [
        ("wave2_r256", "candidate", {**force_env, "GLCUDA_R256": "1"}),
        ("wave3_grid2d", "candidate", {**force_env, "GLCUDA_GRID2D": "1"}),
    ]
    arm_map = {label: (build, env) for label, build, env in arms}

    def percentile(values, q):
        values = sorted(values)
        if not values:
            return None
        index = (len(values) - 1) * q
        lo, hi = math.floor(index), math.ceil(index)
        if lo == hi:
            return values[lo]
        return values[lo] * (hi - index) + values[hi] * (index - lo)

    def session_prefill(path, expected_iters=MEASURE_ITERS):
        data = json.loads(path.read_text(encoding="utf-8"))
        engine_blob = json.dumps(data.get("engine", {}), sort_keys=True).lower()
        if "glcuda" not in engine_blob:
            raise RuntimeError(f"Session did not record glcuda engine: {data.get('engine')}")
        iterations = data.get("measurements", {}).get("iterations", [])
        samples = []
        prompt_counts = []
        for item in iterations:
            ms = float(item.get("prefill_ms", 0.0))
            ntok = int(item.get("prompt_tokens", 0))
            if ms > 0 and ntok > 0:
                samples.append(ntok * 1000.0 / ms)
                prompt_counts.append(ntok)
        if len(samples) != expected_iters:
            raise RuntimeError(f"Expected {expected_iters} prefill samples, found {len(samples)}")
        if len(set(prompt_counts)) != 1:
            raise RuntimeError(f"Prompt token count changed within a session: {prompt_counts}")
        return {
            "samples": samples,
            "prompt_tokens": prompt_counts[0],
            "p50": percentile(samples, 0.50),
            "p90": percentile(samples, 0.90),
            "p99": percentile(samples, 0.99),
        }

    PROD_RECORDS = []
    for repeat in range(PRODUCTION_REPEATS):
        rotated = arms[repeat % len(arms):] + arms[:repeat % len(arms)]
        for label, build_arm, extra_env in rotated:
            archive = RESULTS / f"glbench-{repeat}-{label}.json"
            cmd = [
                BINS[build_arm]["glbench"], "run",
                "--engine", "glcuda",
                "--model", model,
                "--prompt", FIXED_PROMPT,
                "--tokens", "1",
                "--cold-iters", str(COLD_ITERS),
                "--warmup", str(WARMUP_ITERS),
                "--iters", str(MEASURE_ITERS),
                "--temperature", "0",
                "--seed", "42",
                "--kind", "prefill",
                "--out", archive,
            ]
            p = run(cmd, cwd=BASE_DIR if build_arm == "base" else CAND_DIR,
                    env={"CUDA_VISIBLE_DEVICES": "0", **extra_env}, timeout=14400, check=False)
            save_log(f"glbench-{repeat}-{label}.log", p)
            if p.returncode:
                raise SystemExit(f"glbench {label} repeat {repeat} failed; see archived log.")
            hay = p.stdout + "\n" + p.stderr
            if FORCE_Q8_FOR_PTX and "GLCUDA_FORCE_Q8:" not in hay:
                raise SystemExit(f"glbench {label} did not confirm the forced-Q8 arm.")
            if label == "wave2_r256" and "r256 prefill GEMM enabled" not in hay:
                raise SystemExit("Wave 2 arm did not confirm GLCUDA_R256 dispatch.")
            if label == "wave3_grid2d":
                if "2-D token-grid prefill GEMM enabled" not in hay:
                    raise SystemExit("Wave 3 arm did not confirm GLCUDA_GRID2D dispatch.")
                if "r256 prefill GEMM enabled" in hay:
                    raise SystemExit("Wave 3 arm accidentally enabled r256; A/B is invalid.")
            stats = session_prefill(archive)
            rec = {"repeat": repeat, "arm": label, "archive": str(archive), **stats}
            PROD_RECORDS.append(rec)
            print(f"{label:15s} repeat {repeat}: "
                  f"P50 {stats['p50']:.1f}, P90 {stats['p90']:.1f}, P99 {stats['p99']:.1f} tok/s")

    PROD_SUMMARY = []
    for label, _, _ in arms:
        rows = [x for x in PROD_RECORDS if x["arm"] == label]
        PROD_SUMMARY.append({
            "arm": label,
            "session_p50_median": statistics.median(x["p50"] for x in rows),
            "session_p50_min": min(x["p50"] for x in rows),
            "session_p50_max": max(x["p50"] for x in rows),
            "sessions": len(rows),
        })

    print("\nProduction summary:")
    for row in PROD_SUMMARY:
        print(row)

    wave2_rows = [x for x in PROD_RECORDS if x["arm"] == "wave2_r256"]
    wave3_rows = [x for x in PROD_RECORDS if x["arm"] == "wave3_grid2d"]
    paired_deltas = []
    for repeat in range(PRODUCTION_REPEATS):
        a = next(x for x in wave2_rows if x["repeat"] == repeat)
        b = next(x for x in wave3_rows if x["repeat"] == repeat)
        paired_deltas.append(b["p50"] / a["p50"] - 1.0)
    wave2_tps = statistics.median(x["p50"] for x in wave2_rows)
    wave3_tps = statistics.median(x["p50"] for x in wave3_rows)
    median_delta = wave3_tps / wave2_tps - 1.0
    WAVE3_DECISION = {
        "baseline_arm": "wave2_r256", "candidate_arm": "wave3_grid2d",
        "baseline_tps": wave2_tps, "candidate_tps": wave3_tps,
        "median_delta": median_delta, "paired_session_deltas": paired_deltas,
        "correctness_green": True,
        "reproduced_at_5pct": all(x >= 0.05 for x in paired_deltas),
        "retain": median_delta >= 0.05 and all(x >= 0.05 for x in paired_deltas),
    }
    print("\nWave 3 decision:")
    print(json.dumps(WAVE3_DECISION, indent=2))

    # One separate diagnostic run captures event-based stage telemetry. It is
    # never folded into the decision-grade A/B distribution above.
    candidate_labels = ["wave2_r256", "wave3_grid2d"]
    best_label = max(
        candidate_labels,
        key=lambda name: next(r["session_p50_median"] for r in PROD_SUMMARY if r["arm"] == name),
    )
    best_build, best_env = arm_map[best_label]
    telemetry_archive = RESULTS / f"telemetry-{best_label}.json"
    telemetry_cmd = [
        BINS[best_build]["glbench"], "run",
        "--engine", "glcuda", "--model", model, "--prompt", FIXED_PROMPT,
        "--tokens", "1", "--cold-iters", "0", "--warmup", "1",
        "--iters", "1", "--temperature", "0", "--seed", "42",
        "--kind", "prefill", "--out", telemetry_archive,
    ]
    tp = run(telemetry_cmd, cwd=CAND_DIR,
             env={"CUDA_VISIBLE_DEVICES": "0", **best_env, "GLCUDA_TELEMETRY": "1"},
             timeout=14400, check=False)
    save_log(f"telemetry-{best_label}.log", tp)
    if tp.returncode:
        raise SystemExit(f"Telemetry run failed for {best_label}.")
    telemetry_json = json.loads(telemetry_archive.read_text(encoding="utf-8"))
    prefill_telemetry = (telemetry_json.get("telemetry") or {}).get("prefill") or {}
    stages = prefill_telemetry.get("stages") or []
    if not stages:
        raise SystemExit("GLCUDA_TELEMETRY produced no prefill stages.")
    profile_stats = session_prefill(telemetry_archive, expected_iters=1)
    gemm_names = {"qkv", "attn_out", "ffn_gate_up", "ffn_down"}
    stage_total_ms = sum(float(s.get("total_ms", 0.0)) for s in stages)
    gemm_ms = sum(float(s.get("total_ms", 0.0)) for s in stages if s.get("name") in gemm_names)
    gemm_share = gemm_ms / stage_total_ms if stage_total_ms > 0 else 0.0
    measured_best_tps = next(
        r["session_p50_median"] for r in PROD_SUMMARY if r["arm"] == best_label
    )
    prompt_tokens = profile_stats["prompt_tokens"]
    target_prefill_ms = prompt_tokens * 1000.0 / TARGET_PREFILL_TPS
    measured_prefill_ms = prompt_tokens * 1000.0 / measured_best_tps
    required_speedup = TARGET_PREFILL_TPS / measured_best_tps
    isolated_grid_speedup = 3.28
    optimistic_amdahl = 1.0 / ((1.0 - gemm_share) + gemm_share / isolated_grid_speedup)
    projected_grid_tps = measured_best_tps * optimistic_amdahl
    macs_per_prompt = sum(float(s.get("macs") or 0.0) for s in stages)
    target_tmac_s = (macs_per_prompt / prompt_tokens) * TARGET_PREFILL_TPS / 1e12
    TARGET_ANALYSIS = {
        "best_arm": best_label, "prompt_tokens": prompt_tokens,
        "measured_tps": measured_best_tps, "target_tps": TARGET_PREFILL_TPS,
        "measured_prefill_ms": measured_prefill_ms, "target_prefill_ms": target_prefill_ms,
        "required_speedup": required_speedup, "gemm_share": gemm_share,
        "optimistic_2d_grid_tps": projected_grid_tps,
        "target_tmac_s": target_tmac_s,
        "target_fraction_of_t4_int8": target_tmac_s / T4_INT8_TMAC_S,
        "stages": stages,
        "wave3_decision": WAVE3_DECISION,
    }
    print("\n15k feasibility budget:")
    print(json.dumps(TARGET_ANALYSIS, indent=2))

    for repeat in range(PRODUCTION_REPEATS):
        b = RESULTS / f"glbench-{repeat}-wave2_r256.json"
        c = RESULTS / f"glbench-{repeat}-wave3_grid2d.json"
        p = run([BINS["candidate"]["glbench"], "compare", b, c],
                cwd=CAND_DIR, timeout=600, check=False)
        save_log(f"compare-{repeat}-wave2_r256-vs-wave3_grid2d.log", p)
        if p.returncode:
            raise SystemExit("glbench compare failed for Wave 2 r256 vs Wave 3 grid2d.")
    PROD_OK = True


## 7 · Optional Nsight Compute evidence

Kaggle images and host policies vary. If ncu is installed and hardware performance counters are permitted, the cell captures one filtered MMA launch from each diagnostic binary using supported sections. Permission denial is archived as an explicit limitation; it does not turn diagnostic timings into profiler evidence.


In [ ]:
NCU = {"available": False, "runs": {}}
ncu = shutil.which("ncu")
if not RUN_NCU:
    print("Nsight Compute disabled by configuration.")
elif ncu is None:
    print("Nsight Compute CLI is not installed in this Kaggle image.")
else:
    listed = run([ncu, "--list-sections"], timeout=300, check=False)
    save_log("ncu-list-sections.log", listed)
    available_text = listed.stdout + "\n" + listed.stderr
    wanted = [
        "LaunchStats",
        "Occupancy",
        "SpeedOfLight",
        "WarpStateStats",
        "MemoryWorkloadAnalysis",
        "ComputeWorkloadAnalysis",
    ]
    sections = [name for name in wanted if name in available_text]
    NCU["available"] = True
    NCU["sections"] = sections
    print("Nsight sections:", sections)

    for arm in ("wave2_r256", "wave3_grid2d"):
        report = RESULTS / f"ncu-{arm}"
        cmd = [ncu, "--target-processes", "all",
               "--kernel-name", "regex:.*gl_gemm_mma_q8.*", "--launch-count", "1",
               "--force-overwrite", "--export", report]
        for section in sections:
            cmd += ["--section", section]
        if not sections:
            cmd += ["--set", "basic"]
        _, arm_env = arm_map[arm]
        ncu_archive = RESULTS / f"ncu-session-{arm}.json"
        cmd += [BINS["candidate"]["glbench"], "run", "--engine", "glcuda",
                "--model", model, "--prompt", FIXED_PROMPT, "--tokens", "1",
                "--cold-iters", "0", "--warmup", "0", "--iters", "1",
                "--temperature", "0", "--seed", "42", "--kind", "prefill",
                "--out", ncu_archive]
        p = run(cmd, cwd=CAND_DIR, env={"CUDA_VISIBLE_DEVICES": "0", **arm_env},
                timeout=14400, check=False)
        save_log(f"ncu-{arm}.log", p)
        text = p.stdout + "\n" + p.stderr
        permitted = p.returncode == 0 and "ERR_NVGPUCTRPERM" not in text
        NCU["runs"][arm] = {"returncode": p.returncode, "permitted": permitted}
        print(f"{arm:15s}: exit={p.returncode}, counters={'captured' if permitted else 'unavailable'}")
        if permitted:
            imported = run([ncu, "--import", str(report) + ".ncu-rep",
                            "--page", "details", "--csv"], timeout=1800, check=False)
            save_log(f"ncu-{arm}-details.csv", imported)

## 8 · Package the Wave 3 evidence

The final cell writes a machine-readable manifest, a concise Markdown report, and a zip archive. It does not declare an optimization win: that decision belongs at the wave gate after the production archives have been reviewed and reproduced.


In [ ]:
manifest = {
    "schema": "gwenland.glcuda.t4-ceiling.wave3.fetch.v1",
    "created_utc": dt.datetime.now(dt.timezone.utc).isoformat(),
    "gpu_rows": gpu_rows,
    "cuda_visible_devices": os.environ.get("CUDA_VISIBLE_DEVICES"),
    "repo_url": REPO_URL,
    "base_rev": BASE_REV,
    "candidate_patch_sha256": PATCH_SHA256,
    "candidate_markers": markers,
    "tool_versions": tool_versions,
    "ptxas_ok": bool(globals().get("PTXAS_OK")),
    "ptxas_resources": {k: v["resources"] for k, v in globals().get("PTXAS", {}).items()},
    "correctness_ok": bool(globals().get("CORRECTNESS_OK")),
    "production_ok": bool(globals().get("PROD_OK")),
    "target_prefill_tps": TARGET_PREFILL_TPS,
    "force_q8_for_ptx": FORCE_Q8_FOR_PTX,
    "micro_repeats": MICRO_REPEATS,
    "production_repeats": PRODUCTION_REPEATS,
    "cold_iters": COLD_ITERS,
    "warmup_iters": WARMUP_ITERS,
    "measure_iters": MEASURE_ITERS,
    "model_path": str(model) if globals().get("PROD_OK") else None,
    "model_fetch": globals().get("MODEL_FETCH"),
    "micro_table": globals().get("MICRO_TABLE", []),
    "production_summary": globals().get("PROD_SUMMARY", []),
    "target_analysis": globals().get("TARGET_ANALYSIS"),
    "wave3_decision": globals().get("WAVE3_DECISION"),
    "ncu": globals().get("NCU", {}),
}
(RESULTS / "manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")

report = [
    "# glcuda T4 Ceiling - Wave 3",
    "",
    f"- GPU: {gpu_rows[0]}",
    f"- baseline: {BASE_REV}",
    f"- candidate patch: {PATCH_SHA256}",
    f"- model: {HF_REPO}@{HF_REVISION[:12]}/{HF_FILENAME}",
    f"- model SHA-256: {manifest['model_fetch']['sha256'] if manifest.get('model_fetch') else 'FETCH FAILED'}",
    f"- ptxas: {'PASS' if manifest['ptxas_ok'] else 'FAIL'}",
    f"- hardware correctness: {'PASS' if manifest['correctness_ok'] else 'FAIL'}",
    f"- production glbench: {'COMPLETE' if manifest['production_ok'] else 'PENDING - model fetch or production gate failed'}",
    f"- target: {TARGET_PREFILL_TPS:.0f} prefill tok/s",
    "",
    "## Diagnostic GEMM medians",
    "",
    "| arm | shape | 8x64 us | 4x128 us | 2x256 us |",
    "|---|---|---:|---:|---:|",
]
for row in manifest["micro_table"]:
    report.append(
        f"| {row['arm']} | {row['shape']} | {row['t64']:.1f} | "
        f"{row['t128']:.1f} | {row['t256']:.1f} |"
    )

report += ["", "## Production prefill", "",
           "| arm | median of session P50 tok/s | min | max | sessions |",
           "|---|---:|---:|---:|---:|"]
for row in manifest["production_summary"]:
    report.append(
        f"| {row['arm']} | {row['session_p50_median']:.1f} | "
        f"{row['session_p50_min']:.1f} | {row['session_p50_max']:.1f} | {row['sessions']} |"
    )

if manifest["target_analysis"]:
    a = manifest["target_analysis"]
    report += [
        "", "## 15k feasibility budget", "",
        f"- best measured arm: {a['best_arm']} at {a['measured_tps']:.1f} tok/s",
        f"- required end-to-end speedup: {a['required_speedup']:.2f}x",
        f"- measured/target prompt time: {a['measured_prefill_ms']:.2f} / {a['target_prefill_ms']:.2f} ms",
        f"- measured GEMM stage share: {100*a['gemm_share']:.1f}%",
        f"- optimistic Amdahl projection using the isolated 3.28x grid result: {a['optimistic_2d_grid_tps']:.1f} tok/s",
        f"- target linear-layer demand: {a['target_tmac_s']:.2f} TMAC/s ({100*a['target_fraction_of_t4_int8']:.1f}% of 65 TMAC/s)",
    ]

report += [
    "",
    "## Wave 3 gate",
    "",
    (f"- median delta: {100*manifest['wave3_decision']['median_delta']:.2f}%" if manifest.get("wave3_decision") else "- pending"),
    (f"- paired session deltas: {[round(100*x, 2) for x in manifest['wave3_decision']['paired_session_deltas']]}%" if manifest.get("wave3_decision") else "- pending"),
    (f"- verdict: {'RETAIN' if manifest['wave3_decision']['retain'] else 'REJECT/HOLD'}" if manifest.get("wave3_decision") else "- verdict: PENDING"),
    "",
    "## Interpretation rule",
    "",
    "Retain a PTX candidate only after production glbench improves by at least 5%, "
    "the result reproduces in two sessions, correctness stays green, and profiler "
    "evidence does not reveal spills or an occupancy regression that invalidates the comparison.",
    "",
    "Microbenchmark rows are diagnostic only.",
]
(RESULTS / "WAVE3_REPORT.md").write_text("\n".join(report), encoding="utf-8")

archive_base = WORK / "glcuda_t4_ceiling_wave3_fetch_results"
archive = Path(shutil.make_archive(str(archive_base), "zip", root_dir=RESULTS))
print(f"Results directory: {RESULTS}")
print(f"Download archive: {archive}")
print(f"Archive size: {archive.stat().st_size / 1e6:.2f} MB")
print("\n" + "\n".join(report[:30]))
